# Training-only signal audit

This read-only audit checks support for a proposed V5 experiment. It uses transaction Delta version 1 and excludes every timestamp on or after 2025-07-01. It does not train a model or inspect final-test labels.

The internal periods are fit (before 2025-01-01), candidate selection (January–March 2025), and operating-point selection (April–June 2025). These are retrospective diagnostics, not fresh holdouts: earlier models already trained on these rows.

Output contains counts by public transaction categories, hour, and month, without customer, transaction, merchant, session, or IP identifiers. UTC matches the existing experiment convention; source timezone is still unconfirmed.

In [ ]:
import json
spark.conf.set("spark.sql.session.timeZone", "UTC")
sql = "WITH training AS (\n  SELECT\n    CASE WHEN transaction_date < TIMESTAMP '2025-01-01' THEN 'fit'\n         WHEN transaction_date < TIMESTAMP '2025-04-01' THEN 'selection'\n         ELSE 'operating_point' END AS audit_partition,\n    transaction_date, is_fraud, currency, channel, transaction_country,\n    transaction_type, transaction_category, merchant_category\n  FROM workspace.bank_silver.transactions VERSION AS OF 1\n  WHERE transaction_date < TIMESTAMP '2025-07-01'\n    AND is_fraud IS NOT NULL\n), expanded AS (\n  SELECT audit_partition, CAST(is_fraud AS INT) AS label,\n         feature, COALESCE(NULLIF(TRIM(bucket), ''), '__missing__') AS bucket\n  FROM training\n  LATERAL VIEW STACK(9,\n    'all', 'all',\n    'currency', CAST(currency AS STRING),\n    'channel', CAST(channel AS STRING),\n    'transaction_country', CAST(transaction_country AS STRING),\n    'transaction_type', CAST(transaction_type AS STRING),\n    'transaction_category', CAST(transaction_category AS STRING),\n    'merchant_category', CAST(merchant_category AS STRING),\n    'hour', CAST(HOUR(transaction_date) AS STRING),\n    'year_month', DATE_FORMAT(transaction_date, 'yyyy-MM')\n  ) categories AS feature, bucket\n)\nSELECT audit_partition, feature, bucket, COUNT(*) AS rows,\n       SUM(label) AS fraud, AVG(label) AS fraud_rate\nFROM expanded\nGROUP BY audit_partition, feature, bucket\nORDER BY audit_partition, feature, bucket"
rows = spark.sql(sql).collect()
assert len(rows) <= 5000, "Unexpected cardinality; stop before exporting aggregates"
summary = {"source_table": "workspace.bank_silver.transactions", "delta_version": 1, "training_end_exclusive": "2025-07-01", "validation_used": False, "final_test_used": False, "source_tables_modified": False, "row_level_records_exported": False, "groups": [row.asDict() for row in rows]}
print(json.dumps(summary, default=str, indent=2, allow_nan=False))


## Recorded findings — SQL API execution, 2026-10-03

Equivalent aggregate SQL returned 205 groups. The fitting period has 2,271,707 transactions / 2,296 fraud labels; candidate selection has 356,361 / 344; threshold selection has 366,529 / 374. Totals reconcile to 2,994,597 transactions / 3,014 fraud labels.

The largest category/hour rates among groups with at least 10,000 transactions and 20 fraud labels were 0.1285% (fit), 0.1616% (selection), and 0.1553% (threshold period). These descriptive maxima differ across periods and do not establish predictive performance. Similar marginal rates cannot exclude useful interactions or demonstrate that labels are random.

The SQL warehouse timezone was not captured. This notebook sets UTC, but source timestamp semantics remain unconfirmed. The new access recheck still lacked SELECT on digital_events. CatBoost has not been trained.

Full evidence: [audit report](../reports/2026-10-03/train_signal_audit_report.md). These recorded findings come from the SQL API, not a completed Spark execution of this notebook.

## Interpretation

Check positive counts in each internal period before freezing boundaries. Compare category prevalence with the baseline for its own period and inspect monthly stability. A large training-only rate does not prove future utility; small groups are particularly uncertain. Similar marginal rates cannot rule out interactions, and this audit cannot prove that labels are random.

Follow the [V5 improvement specification](../spec/28-09-26-fraud-model/improvement_v5.md). The digital-event branch requires a separate SELECT grant and availability audit. CatBoost, threshold selection, calibration, serving, and promotion have not been executed by this notebook.